# Exercise 10: Hyperparameter Tuning (Solution)
### (010000) Machine Learning - Spring 2026
### Prof. Cheol Jeong, Sejong University

# 1) Load data.

In [19]:
import pandas as pd
import numpy as np

# Read the CSV file uploaded to the current Colab working directory.
# This is the modified housing dataset used in the course.
df = pd.read_csv("Housing price dataset_modified.csv")

# Show the dataset size and the first 5 rows.
print(df.shape)
df.head()

(506, 12)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,36.2


# 2) Select input and target, then make development and test sets.

In [20]:
from sklearn.model_selection import train_test_split

# Use all columns except MEDV as input features.
# Use MEDV as the target variable.
X = df.drop(columns=["MEDV"])
y = df["MEDV"]

# Dataset splitting
# - development set: used for cross-validation
# - test set      : used only once at the very end
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=15
)

print("X_dev shape :", X_dev.shape)
print("X_test shape:", X_test.shape)
print("y_dev shape :", y_dev.shape)
print("y_test shape:", y_test.shape)


X_dev shape : (404, 11)
X_test shape: (102, 11)
y_dev shape : (404,)
y_test shape: (102,)


# 3) Define the pipeline.

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Lasso

# Define one pipeline that contains all steps.
#
# Step 1: StandardScaler
#   - standardize the input features using only the training fold statistics
#     inside each cross-validation split.
#
# Step 2: PolynomialFeatures
#   - generate polynomial and interaction features.
#
# Step 3: Lasso
#   - train lasso regression on the transformed features.
#
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(include_bias=False)),
    ("lasso", Lasso())
])

pipe


Pipeline(steps=[('scaler', StandardScaler()),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('lasso', Lasso())])

# 4) Define the hyperparameter grid.

In [22]:
# Define candidate values for the hyperparameters.
#
# poly__degree controls the order of polynomial features.
# lasso__alpha controls the regularization strength in lasso regression.

# "poly" and "lasso" are user-defined step names.
# "__" combines the step names with the parameter names.
param_grid = {
    "poly__degree": [1, 2, 3],
    "lasso__alpha": [1e-3, 1e-2, 1e-1, 1, 10]
}

# Count the number of candidate combinations.
num_candidates = len(param_grid["poly__degree"]) * len(param_grid["lasso__alpha"])

print("Hyperparameter grid:")
print(param_grid)
print("Number of candidate combinations:", num_candidates)

Hyperparameter grid:
{'poly__degree': [1, 2, 3], 'lasso__alpha': [0.001, 0.01, 0.1, 1, 10]}
Number of candidate combinations: 15


# 5) Run GridSearchCV on the development set.

In [23]:
from sklearn.model_selection import GridSearchCV

# - We use only the development set here.
# - The test set is not used in this step.
# - The development set is split internally into folds for cross-validation.
# - scoring="neg_mean_squared_error" is used because scikit-learn expects
#   larger scores to be better, so MSE is returned with a negative sign.
# - refit=True means that after finding the best hyperparameters, GridSearchCV
#   will fit the best pipeline again on the full development set.
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    refit=True,
    return_train_score=True,    # Save cross-validation result to "cv_results_".
    n_jobs=None                 # No parallel processing
)

grid.fit(X_dev, y_dev)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.775e+02, tolerance: 2.774e+00
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.318e+03, tolerance: 2.728e+00
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.217e+02, tolerance: 2.626e

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('lasso', Lasso())]),
             param_grid={'lasso__alpha': [0.001, 0.01, 0.1, 1, 10],
                         'poly__degree': [1, 2, 3]},
             return_train_score=True, scoring='neg_mean_squared_error')

# 6) Inspect the best hyperparameters and the cross-validation score.

In [24]:
# GridSearchCV stores the best hyperparameters and the best cross-validation score.
# Because the score is negative MSE, convert it back to a positive MSE.
best_degree = grid.best_params_["poly__degree"]
best_alpha = grid.best_params_["lasso__alpha"]
best_cv_mse = -grid.best_score_

print("Best polynomial degree:", best_degree)
print("Best lasso alpha       :", best_alpha)
print("Best 5-fold CV MSE     :", best_cv_mse)

Best polynomial degree: 2
Best lasso alpha       : 0.01
Best 5-fold CV MSE     : 18.973137173665034


# 7) Inspect all candidate results.

In [25]:
# Convert the cross-validation results to a DataFrame.
# Show only several useful columns and sort by rank.
cv_results_df = pd.DataFrame(grid.cv_results_)

show_cols = [
    "param_poly__degree",
    "param_lasso__alpha",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

results_df = cv_results_df[show_cols].copy()

# Convert negative MSE to positive MSE for easier interpretation.
results_df["mean_train_MSE"] = -results_df["mean_train_score"]
results_df["mean_CV_MSE"] = -results_df["mean_test_score"]
results_df["std_CV_MSE"] = results_df["std_test_score"]

results_df = results_df.drop(columns=["mean_train_score", "mean_test_score", "std_test_score"])
results_df = results_df.sort_values(by="rank_test_score").reset_index(drop=True)

results_df

,param_poly__degree,param_lasso__alpha,rank_test_score,mean_train_MSE,mean_CV_MSE,std_CV_MSE
0,2,0.010,1,11.815852,18.973137,4.939138
1,2,0.100,2,14.419745,19.806138,4.924368
2,2,0.001,3,11.454060,20.203491,4.874553
3,3,0.100,4,11.513911,20.670125,6.777678
4,3,1.000,5,22.688610,27.938359,5.364749
5,2,1.000,6,24.781844,29.198221,5.035078
6,1,0.010,7,29.342745,31.376293,4.769860
7,1,0.001,8,29.337021,31.383198,4.757085
8,1,0.100,9,29.805731,31.988357,5.312116
9,1,1.000,10,36.473923,38.516074,8.568745


# 8) Show the best pipeline.

In [26]:
# Because refit=True, GridSearchCV has already trained the best pipeline
# again on the full development set.
best_model = grid.best_estimator_

# 9) Evaluate the final chosen model once on the test set.

In [27]:
from sklearn.metrics import mean_squared_error

# Evaluate the final chosen model only once on the test set.
# The test set must stay untouched until the very end.
y_test_pred = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)

print("Final chosen degree:", best_degree)
print("Final chosen alpha :", best_alpha)
print("Final test MSE     :", test_mse)

Final chosen degree: 2
Final chosen alpha : 0.01
Final test MSE     : 19.03806386493796
